# autoMRE — benchmark on Colab / Kaggle

Reduces a real Python project to a minimal reproduction of one failing
test, then scores it with the Gistify metric (Execution Fidelity).

**Read this before spending a session on it.** The benchmark is CPU-bound
in `pytest` subprocesses. Colab's free tier gives 2 vCPU and Kaggle 4,
against 10 on the laptop these numbers were measured on — so this will
most likely be **slower** than running locally. Use it to leave your own
machine free, to reproduce a number somewhere neutral, or to run a long
ablation unattended. Not to make one run finish sooner.

Kaggle needs **Internet enabled** (Settings → Internet) for the clone and
the pip installs. Anything written under `/kaggle/working` is kept as
session output; the results land there if you clone into it, which is
what the next cell does.

## 1. Get the code

`master` is the trunk and the only branch these instructions cover.

In [ ]:
BRANCH = 'master'

!git clone --quiet --branch $BRANCH https://github.com/i-shantt/autoMRE.git
%cd automre
!git log --oneline -3

## 2. What are we running on?

Reduction here is sequential — one task at a time, because each task
`pip install -e`s into a shared venv and parallel tasks would clobber
each other. Core count therefore affects very little; single-core speed
is what sets the wall clock.

In [ ]:
!python evaluation/cloud_bench.py --summary  # no results yet; prints the host check
import os; print('logical CPUs:', os.cpu_count())

## 3. Run the benchmark

One task per process, each result written separately — so a disconnect
does not cost you the whole run, and re-running this cell continues from
where it stopped rather than starting over.

Start with `--only requests` (four tasks, the fast repo) to confirm the
session survives. Drop the flag for all ten.

Rough local costs, for planning: requests ~3.6 min/task, flask ~20
min/task, tomlkit **26–177 min/task**. Expect worse here.

In [ ]:
!python evaluation/cloud_bench.py --only requests

## 4. The CoveragePruner ablation

The open question: does coverage-based bulk pruning (Phase 4a) earn its
place, or would Phase 4b find the same dead code anyway, just slower?

This runs **both arms of each task, back to back on this host**, and
reports the difference in queries and in output size. It re-runs the
default arm rather than diffing against the numbers in
`results_gistify_threerepo.json`, because those were measured on the
laptop under CPython 3.13 — `lines` and `queries` are deterministic for a
given interpreter, not across interpreters, so a cross-machine diff would
confound the ablation with the environment.

**It doubles the runtime.** Do `--only requests` first (8 runs). Kaggle
sessions are capped at 12 hours, so run flask and tomlkit in separate
sessions — results are per task and per arm, so nothing is lost by
splitting.

In [ ]:
!python evaluation/cloud_bench.py --ablation coverage-prune --only requests

In [ ]:
# Separate sessions for the slow repos:
# !python evaluation/cloud_bench.py --ablation coverage-prune --only flask
# !python evaluation/cloud_bench.py --ablation coverage-prune --only tomlkit

## 5. Results

Compare **queries and line counts**, not wall clock: those are
deterministic for a given interpreter, wall clock is not comparable
across machines.

Reference (M4 MacBook Air, sequential, 10 tasks / 3 repos): 86.53%
aggregate, 95.23% over the code eligible for removal, 10/10 fidelity,
23,326 queries. The paper's best reported execution fidelity is 58.7%.

A fidelity below 10/10 here is a finding worth reporting, not a fluke —
it would mean the pipeline depends on something about the laptop.

In [ ]:
!python evaluation/cloud_bench.py --summary
!python evaluation/cloud_bench.py --summary --ablation coverage-prune

## 6. Reduce your own project

Point it at a directory and the command that reproduces your bug. What
survives is a small tree that still reproduces it.

The command names the test, and that test plus any `conftest.py` above it
are protected — they are the statement of what must stay true, not code
to be reduced.

In [ ]:
# !python automre.py reduce-project /path/to/project \
#     --command 'python -m pytest tests/test_thing.py::test_case -x -q'